In [0]:
storage_account_name = "storage60301575"
storage_account_key = "2hVwV/06JBSdKiOTedzP6WHZpiJSrfZVdwK8psbRCVT8yQKzRponj/dQecMc2DFc/UUG3u/Wocqq+AStijSp5w=="
spark.conf.set(
 f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
 storage_account_key
)
reviews_path = "abfss://processed@storage60301575.dfs.core.windows.net/reviews/"
reviews_df = spark.read.parquet(reviews_path)


In [0]:
gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/features_v1/"
gold_df = spark.read.parquet(gold_path)

gold_df.printSchema()
display(gold_df.limit(5))

root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reviewTime: string (nullable = true)
 |-- review_year: integer (nullable = true)



asin,title,brand,price,reviewerID,overall,summary,reviewText,helpful,reviewTime,review_year
B0058BDFXA,LaCie Rugged Hard Disk Triple 500 GB 7200 rpm USB 3.0 Firewire 800 (2x) Portable Hard Drive 301983,LaCie,119.99,A2RN0V7PRP8JJR,5.0,Awesome Device!!!,"It's is smaller than I thought and lighter too. It came with all the cables you might need for now days I/O technologies. And it's 7200 rpm make it a really handy and portable solution for every file you want. It need to be reformatted if you want it to use it on a Mac. The shipping and handling were the fastest I've ever had in amazon, that's why I totally recommend to buy from this guys.","List(0, 0)","08 14, 2013",2013
B0058BDFXA,LaCie Rugged Hard Disk Triple 500 GB 7200 rpm USB 3.0 Firewire 800 (2x) Portable Hard Drive 301983,LaCie,119.99,A1J3X32Q7MC6VV,5.0,Cool!,I have one at home and one at work and I have travelled with it too! It is great to use.,"List(0, 0)","04 3, 2013",2013
B0058BDFXA,LaCie Rugged Hard Disk Triple 500 GB 7200 rpm USB 3.0 Firewire 800 (2x) Portable Hard Drive 301983,LaCie,119.99,AL4L9RE6FDDZ7,1.0,My experience with LaCie Drives,"I buy hard drives for storing HD video files at a clip of about 2-3 a month and years ago I was sold on these ""rugged"" drives. I soon learned from using these drives that although they look like they will stand up to abuse and the case is stronger than most. Whats more important is that the drives not fail on you during normal use. Unfortunately I have found thru experience that LaCie drives fail the most of the major manufacturers. I recently gave them another chance after a couple of years of avoiding their drives and not much has changed. Sad but true.","List(0, 0)","09 12, 2013",2013
B0058BDFXA,LaCie Rugged Hard Disk Triple 500 GB 7200 rpm USB 3.0 Firewire 800 (2x) Portable Hard Drive 301983,LaCie,119.99,A18IJ164U7LH3R,4.0,So far I like this external hard drive; it works as advertised with decent transfer rates.,"What is it?Basically it’s a rugged style external hard disk drive that has 2 FireWire 800 ports and a USB 3.0 port. With 7200 rpms it makes the device Ideal for transferring large files with quick transfer rates.Specifications: (Note that Transfer rates depend on many factors including your computers CPU)USB 3.0 & 2x FireWire 800 USB 3.0 (USB 2.0 Support) 2x FireWire 800 (FireWire 400 Support) USB 3.0 can transfer up to 5GBs/s (theoretically) which translates to about 110mbs/s with 7200 RPMs USB 2.0 can Transfer up to 480mbs/s (theoretically) which translates to about 30mbs/s with 7200 RPMs FireWire 800 can transfer up to 800mbs/s (theoretically) which translates to about 65mbs/s with 7200 RPMs FireWire 400 can transfer up to 400mbs/s (theoretically) which translates to about 25mbs/s with 7200 RPMsPros:- Relatively small and compact- Ships with both a USB 3.0 and FireWire 800 cable.- Inexpensive for what you get- Rugged, so it can take a beating- 7200 RPMs- USB 3.0 is AwesomeCons:- Not a huge fan of the Orange color- Comes with bloat-ware- The cables that it comes with are super short (basically a ruler and a half)- Device gets hot with prolonged use…My Thoughts:So far I like this external hard drive; it works as advertised with decent transfer rates. Now I’m starting to wonder how I managed on regular USB 2.0 for so long. USB 3.0 is a straight up beast at transferring files compared to pretty much, everything else. So far the LaCie Rugged drive has been surviving my daily travels; I just throw it in my bag and go. Currently I highly recommend this device to anyone who does a lot of editing (Video, Audio, Photo), especially Video. It will also make a good network drive for those who stream a lot of media over their network.Build Quality / Design: 4.5Features: 4.0Price: 4.0Space: 3.5Average (Amazon | Newegg): 3.4Overall19.4/2578View full review on my website [...]","List(0, 1)","09 15, 2013",2013
B0058BDFXA,LaCie Rugged Hard Disk Triple 500 GB 7200 rpm USB 3.0 Firewire 800 (2x) Portable Hard Drive 301983,LaCie,119

In [0]:
print("Number of rows:", gold_df.count())
print("Number of columns:", len(gold_df.columns))

Number of rows: 11811989
Number of columns: 11


In [0]:
from pyspark.sql.functions import col, count, when, length

gold_df.select(
    count(when(col("reviewText").isNull() | (length(col("reviewText")) == 0), True)).alias("missing_reviewText"),
    count(when(col("overall").isNull(), True)).alias("missing_overall")
).show()

+------------------+---------------+
|missing_reviewText|missing_overall|
+------------------+---------------+
|                 0|              0|
+------------------+---------------+



In [0]:
rating_dist = gold_df.groupBy("overall").count().orderBy("overall")
display(rating_dist)

overall,count
1.0,760284
2.0,574392
3.0,994742
4.0,2426851
5.0,7055720


In [0]:
from pyspark.sql.functions import length

gold_df = gold_df.withColumn("review_length_chars", length(col("reviewText")))
display(gold_df.select("review_length_chars"))

review_length_chars
392
88
561
1931
231
523
439
128
269
190


In [0]:
sample_n = 300_000
sample_seed = 42

# Stratified sampling by review_year
year_counts = gold_df.groupBy("review_year").count()
total_count = gold_df.count()

fractions = {
    row["review_year"]: sample_n * (row["count"] / total_count) / row["count"]
    for row in year_counts.collect()
}

df_sampled = gold_df.sampleBy("review_year", fractions=fractions, seed=sample_seed)
df_sampled = df_sampled.limit(sample_n)

print("Sampled rows:", df_sampled.count())

Sampled rows: 299879


In [0]:
df_sampled.printSchema()

print("Original dataset rating distribution:")
gold_df.groupBy("overall").count().orderBy("overall").show()

print("Sampled dataset rating distribution:")
df_sampled.groupBy("overall").count().orderBy("overall").show()

root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reviewTime: string (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_length_chars: integer (nullable = true)

Original dataset rating distribution:
+-------+-------+
|overall|  count|
+-------+-------+
|    1.0| 760284|
|    2.0| 574392|
|    3.0| 994742|
|    4.0|2426851|
|    5.0|7055720|
+-------+-------+

Sampled dataset rating distribution:
+-------+------+
|overall| count|
+-------+------+
|    1.0| 19322|
|    2.0| 14460|
|    3.0| 25537|
|    4.0| 61364|
|    5.0|179196|
+-------+------+



In [0]:
sampled_gold_path = f"abfss://curated@{storage_account_name}.dfs.core.windows.net/features_v1_sampled/"

df_sampled.write.mode("overwrite").parquet(sampled_gold_path)